# Deep Learning with PyTorch · Transfer learning and deployment

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Pretrained ResNet-18

In [ ]:
from torch import nn
from torchvision import models
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(sum(p.numel() for p in model.parameters()))                   # 11 689 512
for name, mod in [('stem', nn.Sequential(model.conv1, model.bn1)), ('layer1', model.layer1), ('layer2', model.layer2), ('layer3', model.layer3), ('layer4', model.layer4)]:
    print(name, sum(p.numel() for p in mod.parameters()))

## 2. Freeze and add a new head

In [ ]:
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(512, 5)
print('trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))
for p in model.layer4.parameters():
    p.requires_grad = True                                           # fine-tuning: last block
print('layer4 unfrozen:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 3. Export: TorchScript and ONNX

In [ ]:
model.eval()
ornek = torch.randn(1, 3, 224, 224)
ts = torch.jit.trace(model, ornek); ts.save('model_ts.pt')
torch.onnx.export(model, ornek, 'model.onnx', input_names=['goruntu'], output_names=['skor'], dynamic_axes={'goruntu': {0: 'N'}})
import os
print(os.path.getsize('model_ts.pt') // 1024, 'KB', os.path.getsize('model.onnx') // 1024, 'KB')

## 4. Dynamic int8 quantisation

In [ ]:
import time
q = torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
x = torch.randn(8, 3, 224, 224)
for name, m in [('fp32', model), ('int8 (Linear)', q)]:
    t = time.perf_counter()
    with torch.no_grad():
        y = m(x)
    print(name, f'{(time.perf_counter() - t) * 1000:.0f} ms', y.shape)
torch.save(q.state_dict(), 'q.pt'); torch.save(model.state_dict(), 'f.pt')
print(os.path.getsize('f.pt') // 1024, 'KB ->', os.path.getsize('q.pt') // 1024, 'KB')

## Exercises
1. Train only the head on your own small 5-class image set (50 images per class).
2. Run the ONNX model with `onnxruntime` and compare the output with PyTorch.
3. Try static (calibrated) quantisation.